# Image Captioning Challenge - V1
## BLIP-2 Baseline Inference Notebook

This notebook performs:
- dataset loading and inspection,
- exploratory visualization,
- BLIP-2 based caption generation,
- submission file creation,
- local proxy FGD score calculation.

In [ ]:
# Cell 0 - Imports, Setup, Drive Mount, Dataset Extraction, and CSV Loading

import os
import warnings
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm import tqdm

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

from numpy import cov, trace, iscomplexobj
from scipy.linalg import sqrtm

from google.colab import drive

warnings.filterwarnings("ignore")

# Pandas display settings
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Mount Google Drive
drive.mount("/content/drive")

# Dataset paths
DRIVE_RAR_PATH = "/content/drive/MyDrive/Image_Captioning/image_captioning_dataset.rar"
EXTRACT_PATH = "/content/dataset"
BASE_DATASET_PATH = os.path.join(EXTRACT_PATH, "image_captioning_dataset")

TRAIN_CSV_PATH = os.path.join(BASE_DATASET_PATH, "train.csv")
TEST_CSV_PATH = os.path.join(BASE_DATASET_PATH, "test.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_DATASET_PATH, "sample_submission.csv")

TRAIN_IMAGE_DIR = os.path.join(BASE_DATASET_PATH, "train", "train")
TEST_IMAGE_DIR = os.path.join(BASE_DATASET_PATH, "test", "test")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Create extraction directory
os.makedirs(EXTRACT_PATH, exist_ok=True)

# Install unrar silently
subprocess.run(
    ["apt-get", "-qq", "install", "-y", "unrar"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Extract dataset silently
subprocess.run(
    ["unrar", "x", "-o+", DRIVE_RAR_PATH, f"{EXTRACT_PATH}/"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Verify dataset paths
assert os.path.exists(TRAIN_CSV_PATH), f"Missing file: {TRAIN_CSV_PATH}"
assert os.path.exists(TEST_CSV_PATH), f"Missing file: {TEST_CSV_PATH}"
assert os.path.exists(SAMPLE_SUB_PATH), f"Missing file: {SAMPLE_SUB_PATH}"
assert os.path.exists(TRAIN_IMAGE_DIR), f"Missing directory: {TRAIN_IMAGE_DIR}"
assert os.path.exists(TEST_IMAGE_DIR), f"Missing directory: {TEST_IMAGE_DIR}"

# Read CSV files
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)
sample_df = pd.read_csv(SAMPLE_SUB_PATH)

print("Setup completed successfully.")
print("Using device:", DEVICE)
print("Train set shape:", train_df.shape)
print("Test set shape:", test_df.shape)
print("Sample submission shape:", sample_df.shape)

print("\nTrain DataFrame Preview:")
print(train_df.head().to_string(index=False))

print("\nTest DataFrame Preview:")
print(test_df.head().to_string(index=False))

print("\nSample Submission Preview:")
print(sample_df.head().to_string(index=False))

In [ ]:
# Cell 1 - Dataset Checks & Caption Length Distribution

# Basic Dataset Checks
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Sample submission columns:", sample_df.columns.tolist())

print("Unique captions:", train_df["caption"].nunique())

train_df["caption_length"] = train_df["caption"].apply(lambda x: len(str(x).split()))
print("Average caption length:", train_df["caption_length"].mean())
print("Min caption length:", train_df["caption_length"].min())
print("Max caption length:", train_df["caption_length"].max())

# Caption Length Distribution
plt.figure(figsize=(10, 5))
sns.histplot(train_df["caption_length"], bins=30, kde=True)
plt.title("Caption Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 2 - Random Training Examples Visualization

import textwrap

def show_random_examples(df, image_dir, n=5):
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 7))

    if n == 1:
        axes = [axes]

    sampled_rows = df.sample(n=min(n, len(df)), random_state=None)

    for ax, (_, row) in zip(axes, sampled_rows.iterrows()):
        img_filename = f"{row['image_id']}.jpg"
        img_path = os.path.join(image_dir, img_filename)

        if not os.path.exists(img_path):
            ax.axis("off")
            ax.set_title(f"Missing:\n{img_filename}", fontsize=10)
            continue

        image = Image.open(img_path).convert("RGB")
        ax.imshow(image)
        ax.axis("off")

        full_caption = str(row["caption"])
        wrapped_caption = textwrap.fill(full_caption, width=35)
        ax.set_title(wrapped_caption, fontsize=10)

    plt.tight_layout()
    plt.show()

show_random_examples(train_df, TRAIN_IMAGE_DIR, n=5)
show_random_examples(train_df, TRAIN_IMAGE_DIR, n=5)

In [ ]:
# Cell 3 - Load BLIP-2 Processor and Model

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)

print("BLIP-2 model and processor loaded successfully.")

In [ ]:
# Cell 4 - Optional Caption Preprocessing Column

train_df["caption_proc"] = train_df["caption"].apply(
    lambda x: "<start> " + str(x).strip() + " <end>"
)

display(train_df[["image_id", "caption", "caption_proc"]].head())

In [ ]:
# Cell 5 - Single Test Image Inference Example

sample_row = test_df.sample(1).iloc[0]
sample_img_id = str(sample_row["image_id"])
sample_img_filename = f"{sample_img_id}.jpg"
sample_img_path = os.path.join(TEST_IMAGE_DIR, sample_img_filename)

image = Image.open(sample_img_path).convert("RGB")

inputs = processor(images=image, return_tensors="pt")
inputs = {
    k: v.to(DEVICE, dtype=torch.float16) if v.dtype.is_floating_point else v.to(DEVICE)
    for k, v in inputs.items()
}

with torch.no_grad():
    generated_ids = blip_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

caption = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title(f"ID: {sample_img_id}\nGenerated Caption:\n{caption}", fontsize=10)
plt.tight_layout()
plt.show()

print("Generated caption:", caption)

In [ ]:
# Cell 6 - Generate Captions for Test Set

submission = []

MIN_WORDS = 8
PRIMARY_MIN_NEW_TOKENS = 10
FALLBACK_MIN_NEW_TOKENS = 12
MAX_NEW_TOKENS = 30

def clean_caption(text):
    text = text.replace("\n", " ").replace("\r", " ").strip()
    text = " ".join(text.split())
    return text

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img_id = str(row["image_id"])
    img_filename = f"{img_id}.jpg"
    img_path = os.path.join(TEST_IMAGE_DIR, img_filename)

    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        print(f"Image not found or corrupted: {img_path}")
        continue

    inputs = processor(images=image, return_tensors="pt")
    inputs = {
        k: v.to(DEVICE, dtype=torch.float16) if v.dtype.is_floating_point else v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = blip_model.generate(
            **inputs,
            min_new_tokens=PRIMARY_MIN_NEW_TOKENS,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=2
        )

    caption = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    caption = clean_caption(caption)

    if len(caption.split()) < MIN_WORDS:
        with torch.no_grad():
            generated_ids = blip_model.generate(
                **inputs,
                min_new_tokens=FALLBACK_MIN_NEW_TOKENS,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=5,
                do_sample=False,
                no_repeat_ngram_size=2,
                length_penalty=1.1
            )

        caption = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        caption = clean_caption(caption)

    submission.append({
        "image_id": row["image_id"],
        "caption": caption
    })

submission_df = pd.DataFrame(submission)
print("Submission shape:", submission_df.shape)
print(submission_df.head().to_string(index=False))

In [ ]:
# Cell 7 - Save Submission File

submission_output_path = "/content/submission_v1.csv"
submission_df.to_csv(submission_output_path, index=False)

print(f"Submission file saved to: {submission_output_path}")
display(pd.read_csv(submission_output_path).head())

In [ ]:
# Cell 8 - Load Embedding Model for Local FGD Calculation

# Install Sentence Transformers
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

gte_model = SentenceTransformer("thenlper/gte-small")
print("Embedding model loaded successfully.")

In [ ]:
# Cell 9 - FGD Calculation Function

def calculate_fgd(solution_embed: np.ndarray, submission_embed: np.ndarray) -> float:
    fgd_list = []

    for sol_emb_sample, sub_emb_sample in zip(solution_embed, submission_embed):
        sol_emb_sample_rshaped = sol_emb_sample.reshape((1, 384))
        sub_emb_sample_rshaped = sub_emb_sample.reshape((1, 384))

        e1 = np.concatenate([sol_emb_sample_rshaped, sol_emb_sample_rshaped])
        e2 = np.concatenate([sub_emb_sample_rshaped, sub_emb_sample_rshaped])

        mu1, sigma1 = e1.mean(axis=0), cov(e1, rowvar=False)
        mu2, sigma2 = e2.mean(axis=0), cov(e2, rowvar=False)

        ssdiff = np.sum((mu1 - mu2) ** 2.0)
        covmean = sqrtm(sigma1.dot(sigma2))

        if iscomplexobj(covmean):
            covmean = covmean.real

        fgd = ssdiff + trace(sigma1 + sigma2 - 2.0 * covmean)
        fgd_list.append(fgd)

    return float(np.mean(fgd_list))

# Calculate Local Proxy FGD Score

gt_df = pd.read_csv(TRAIN_CSV_PATH)
pred_df = pd.read_csv(submission_output_path)

N = min(len(gt_df), len(pred_df))
gt_captions = gt_df["caption"].values[:N]
pred_captions = pred_df["caption"].values[:N]

gt_embed = gte_model.encode(list(gt_captions), batch_size=64, show_progress_bar=True)
pred_embed = gte_model.encode(list(pred_captions), batch_size=64, show_progress_bar=True)

fgd_score = calculate_fgd(gt_embed, pred_embed)

print(f"Local Proxy FGD Score: {fgd_score:.4f}")

In [ ]:
# Cell 10 - Preview Final Outputs and Random 5x5 Visualization

import textwrap

print("Submission preview:")
print(submission_df.head(10).to_string(index=False))

print("\nNumber of generated captions:", len(submission_df))
print("Expected number of test images:", len(test_df))

def visualize_generated_captions_grid(submission_df, image_dir, n_rows=5, n_cols=5):
    total_images = n_rows * n_cols
    sampled_df = submission_df.sample(n=min(total_images, len(submission_df))).reset_index(drop=True)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 22))
    axes = axes.flatten()

    fig.suptitle(
        "Random Generated Caption Samples from the Test Set",
        fontsize=20,
        y=0.995
    )

    for ax, (_, row) in zip(axes, sampled_df.iterrows()):
        image_id = str(row["image_id"])
        caption = str(row["caption"]).replace("\n", " ").replace("\r", " ").strip()
        caption = " ".join(caption.split())

        img_filename = f"{image_id}.jpg"
        img_path = os.path.join(image_dir, img_filename)

        if not os.path.exists(img_path):
            ax.axis("off")
            ax.set_title(f"Missing Image\nID: {image_id}", fontsize=10)
            continue

        image = Image.open(img_path).convert("RGB")
        ax.imshow(image)
        ax.axis("off")

        wrapped_caption = textwrap.fill(caption, width=35)
        ax.set_title(f"ID: {image_id}\n{wrapped_caption}", fontsize=9)

    for ax in axes[len(sampled_df):]:
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

visualize_generated_captions_grid(
    submission_df=submission_df,
    image_dir=TEST_IMAGE_DIR,
    n_rows=5,
    n_cols=5
)

# Experiment Summary

## Description
- V1 baseline image captioning notebook with BLIP-2.
- BLIP-2 is a vision-language model that generates natural-language captions from images.
- Used to generate captions for the test set.

## Methodology
- Inspected caption statistics and visualized sample training images.
- Generated captions with BLIP-2.
- Cleaned outputs and reduced overly short captions.
- Measured performance with a local proxy FGD score.

## Score
- Local Proxy FGD Score: **0.4707**

## Notes
- FGD (Fréchet GTE Distance) measures how close the generated caption distribution is to the reference caption distribution in embedding space.
- Local development score only.